# TODO:

## None

In [113]:
import numpy as np
import random
import matplotlib.pyplot as plt

class DLA:
    def __init__(self, grid_size, stickiness, max_particles):
        """
        Initialization function:
            Common python convention is to use __init__ for initialization of class attributes.
            This function sets up the grid, initializes the cluster with a seed particle, and prepares parameters for the simulation.
        """
        # Initialize parameters
        self.grid_size      = grid_size                         # Length and width of the square grid
        self.center         = grid_size // 2                    # Center of the grid
        self.max_steps      = self.grid_size * 10               # Prevent infinite loops (particles wandering forever)
        self.stickiness     = stickiness                        # Probability of sticking when adjacent
        self.max_particles  = max_particles                     # Total number of particles to add to the cluster 

        # Initialize grid
        self.grid           = np.zeros((grid_size, grid_size))  # Create the square grid (matrix) 
        self.grid[self.center, self.center] = 1                 # Used floor division to ensure integer index. Put seed in the center of grid
        
        # Initialize cluster
        self.cluster_particles  = [(self.center, self.center)]  # Creates a tuple list to store the coords of particles in the cluster
        self.cluster_radius     = 1                             # Initial radius of the cluster (distance from center to farthest particle)

    def new_particle(self):
        """
        Introduce a new particle on a circle around the cluster
        Use random angle to place the particle on the circumference of a circle 
        Use a radius that is larger than the current cluster radius
        """
        theta   = random.uniform(0, 2*np.pi)        # Random angle
        r       = self.cluster_radius + 5           # Starting position is outside the cluster, so the particle can diffuse
        
        x = int(self.center + r * np.cos(theta))    # The int() function forces the resulting vector to be an integer value
        y = int(self.center + r * np.sin(theta))    # Since we are using computers and walking particles on a lattice, we need ints
        
        return x, y

    def step(self, x, y):
        """
        I have learned that python has a great built-in "random" library
        This simple function simply takes the current pos, and randomly steps it
        """
        direction = random.choice([(1,0),(-1,0),(0,1),(0,-1)])
        return x + direction[0], y + direction[1]
    
    def walk(self):
        """
        Was initially apart of the simulation method, but I broke it into a seperate method
        This function simulates the random walk of a single particle until it either sticks to the cluster or wanders too far away.
        """
        x, y = self.new_particle()  # Start a new particle
        steps = 0                   # Step counter to track wandering

        while steps < self.max_steps:
            x, y = self.step(x, y)  # Diffuse
            steps += 1              # Track steps

            """ 
            If particle wonders too far away from cluster: (e.g. if it tries to leave the grid)
                Simulate it "leaving" and another particle "entering" 
                by breaking the loop and starting a new one with a new particle.
            """
            r = np.sqrt((x - self.center)**2 + (y - self.center)**2)
            if r > (self.center - 2):       # The spawn distance was radius*2... don't let it leave the grid, prevent edge cases
                x, y = self.new_particle()  # Respawn a new particle, the code will automatically stop tracking the old one and start tracking the new one
                steps = 0                   # Reset step counter for new particle

            if self.is_on_cluster(x, y):
                if random.random() < self.stickiness:       # This is where stickiness/probability matters. btw, random.random() generates a float between 0 and 1
                    return x, y             # The particle stuck: here

        # Else, return None, indicating the particle wandered without sticking
        return None

    def is_on_cluster(self, x, y):
        """
        Short function that checks surrounding positions for cluster particles.
        Might modify later to check for multiple contact points
        """
        surrounding_pts = [(1,0),(-1,0),(0,1),(0,-1)]   # Steps needed to get from curr_pos to all surrounding_pts
        for dx, dy in surrounding_pts:                  # Quick loop through all surrounding points
            if self.grid[x+dx, y+dy] > 0:              # Check if any surrounding point is part of the cluster (grid value of 1)
                return True
        return False

    def update_cluster_radius(self, x, y):
        r = np.sqrt((x - self.center)**2 + (y - self.center)**2)    # Simple distance formula to find new rad
        if r > self.cluster_radius:                                 # Check if radius needs updated
            self.cluster_radius = r                                 # Update as needed   

    def simulate(self):
        """
        Main loop for simulating the DLA process
        """
        particles_added = 1                     # From the seed
        
        while particles_added < self.max_particles:
            result = self.walk()            # Simulate the random walk for a single particle
            
            if result is not None:
                # The particle stuck:
                x, y = result                           # Unpack the coordinates of the stuck particle
                self.grid[x, y] = particles_added       # Update grid 
                self.cluster_particles.append((x, y))   # Add particle to cluster list
                self.update_cluster_radius(x, y)        # Update cluster radius if needed
                particles_added += 1                    # Increment the count of particles in the cluster
            
            if particles_added % 1000 == 0:
                self.plot()                             # Plot the cluster after each new particle sticks (can be commented out for faster simulation)

    def plot(self):
        # https://matplotlib.org/stable/api/figure_api.html
        plt.figure(figsize=(8,8))                                           # This simply adjusts the size of the plot (width, height) in in.
       
        # https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html
        plt.imshow(self.grid + 1, cmap='inferno', origin='lower')           # This is the main plotting. It takes the value of each cell in the grid and maps it to a color (logrithmicly: hince the +1)
       
        # https://matplotlib.org/stable/api/colorbar_api.html
        plt.colorbar(label='Particle Deposition Order (Growth Time Step)')  # This just adds the colorbar for clarity, showing the age of particles in the cluster (log scale)

        plt.title('Diffusion Limited Aggregation Cluster Growth')
        
        # https://matplotlib.org/stable/api/axis_api.html
        plt.axis('off')                                                     # Turns looks better

In [ ]:
# Simulation parameters
grid_size       = 300
stickiness      = .5
max_particles   = 5000

# Create simulation object, run the simulation, and plot the results
sim = DLA(grid_size, stickiness, max_particles)
sim.simulate()
# sim.plot()